# Case Study 4: Time Series Forecasting with Genetic Algorithms

## 🎯 Learning Objectives

In this case study, you will learn:

1. **Time Series Fundamentals**
   - Time series components (trend, seasonality, noise)
   - Feature engineering for time series
   - Train/validation/test splits for temporal data
   - Evaluation metrics (MAE, RMSE, MAPE)

2. **GA for Time Series Optimization**
   - Lag feature selection
   - Window size optimization
   - Model hyperparameter tuning
   - Ensemble forecasting

3. **Practical Applications**
   - Optimizing ARIMA parameters
   - Feature engineering with GA
   - Multi-step forecasting
   - Comparing with baseline methods

---

## 📚 Background: Time Series Forecasting

### What is Time Series Forecasting?

**Time series forecasting** predicts future values based on historical observations. Applications include:
- Stock price prediction
- Demand forecasting
- Weather prediction
- Energy consumption
- Sales forecasting

### Key Components

1. **Trend**: Long-term increase/decrease
2. **Seasonality**: Repeating patterns (daily, weekly, yearly)
3. **Cyclic**: Long-term oscillations
4. **Residual**: Random noise

### Feature Engineering for Time Series

**Lag features**: Previous time step values
```python
lag_1 = series[t-1]  # Value 1 step ago
lag_7 = series[t-7]  # Value 7 steps ago (weekly seasonality)
```

**Rolling statistics**: Moving averages, std dev
```python
rolling_mean_7 = series[t-7:t].mean()
rolling_std_7 = series[t-7:t].std()
```

**Datetime features**: Day of week, month, quarter
```python
day_of_week = date.weekday()
month = date.month
```

### Why Use GA?

**Challenges**:
- Which lag features to use?
- Optimal window sizes for rolling features?
- Best model hyperparameters?
- How to combine multiple forecasts?

**GA Solution**:
- Search over feature combinations
- Optimize window sizes and lags
- Tune model parameters
- All simultaneously!

---

## 🧬 Problem Formulation

### Chromosome Encoding

We'll optimize feature engineering for time series forecasting:

```
Chromosome = [lag_selection | rolling_windows | model_hyperparams]
             [max_lag bits  | n_windows reals | k reals        ]
```

**Example** (max_lag=10, 3 rolling features):
```
[1, 0, 1, 1, 0, 0, 1, 0, 0, 1, | 0.2, 0.5, 0.8 | 0.6, 0.3]
 ^lags: 1,3,4,7,10^              ^window sizes^   ^model params^
```

Features created:
- lag_1, lag_3, lag_4, lag_7, lag_10
- rolling_mean_7, rolling_mean_14, rolling_mean_28 (windows scaled from genes)
- Model: RandomForest(n_estimators=scaled, max_depth=scaled)

### Fitness Function

```python
fitness = -validation_RMSE - complexity_penalty
```

Where:
- `validation_RMSE`: Root mean squared error on validation set
- `complexity_penalty`: Penalize too many features

---

## 1️⃣ Setup and Data Generation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")

In [ ]:
def generate_synthetic_time_series(n_points=500, trend=0.05, seasonality_period=30, 
                                   noise_level=0.1, random_state=42):
    """
    Generate synthetic time series with trend, seasonality, and noise.
    
    Arguments:
    n_points -- number of time points
    trend -- linear trend coefficient
    seasonality_period -- period of seasonal component
    noise_level -- standard deviation of noise
    random_state -- random seed
    
    Returns:
    series -- time series values
    dates -- datetime index
    """
    np.random.seed(random_state)
    
    t = np.arange(n_points)
    
    # Components
    trend_component = trend * t
    seasonal_component = 10 * np.sin(2 * np.pi * t / seasonality_period)
    noise = np.random.normal(0, noise_level * 10, n_points)
    
    # Combined series
    series = 100 + trend_component + seasonal_component + noise
    
    # Create datetime index
    dates = pd.date_range(start='2020-01-01', periods=n_points, freq='D')
    
    return series, dates


# Generate data
series, dates = generate_synthetic_time_series(n_points=500, trend=0.05, 
                                                seasonality_period=30, noise_level=0.15)

# Create DataFrame
df = pd.DataFrame({'date': dates, 'value': series})
df.set_index('date', inplace=True)

print(f"Time series length: {len(df)}")
print(f"Date range: {df.index[0]} to {df.index[-1]}")
print(f"\nFirst 5 values:")
print(df.head())
print(f"\nLast 5 values:")
print(df.tail())

In [ ]:
# Visualize time series
plt.figure(figsize=(14, 5))
plt.plot(df.index, df['value'], linewidth=1.5, color='steelblue')
plt.xlabel('Date', fontsize=11)
plt.ylabel('Value', fontsize=11)
plt.title('Synthetic Time Series (Trend + Seasonality + Noise)', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean: {df['value'].mean():.2f}")
print(f"Std: {df['value'].std():.2f}")
print(f"Min: {df['value'].min():.2f}")
print(f"Max: {df['value'].max():.2f}")

## 2️⃣ Feature Engineering for Time Series

In [ ]:
def create_time_series_features(df, selected_lags, rolling_windows, add_datetime_features=True):
    """
    Create features from time series.
    
    Arguments:
    df -- DataFrame with 'value' column and datetime index
    selected_lags -- list of lag values to create (e.g., [1, 3, 7])
    rolling_windows -- list of window sizes for rolling stats (e.g., [7, 14])
    add_datetime_features -- whether to add day/month/quarter features
    
    Returns:
    features_df -- DataFrame with all features
    """
    features_df = df.copy()
    
    # Lag features
    for lag in selected_lags:
        features_df[f'lag_{lag}'] = features_df['value'].shift(lag)
    
    # Rolling statistics
    for window in rolling_windows:
        features_df[f'rolling_mean_{window}'] = features_df['value'].rolling(window=window).mean()
        features_df[f'rolling_std_{window}'] = features_df['value'].rolling(window=window).std()
    
    # Datetime features
    if add_datetime_features:
        features_df['day_of_week'] = features_df.index.dayofweek
        features_df['day_of_month'] = features_df.index.day
        features_df['month'] = features_df.index.month
        features_df['quarter'] = features_df.index.quarter
    
    # Drop rows with NaN (due to lags and rolling)
    features_df.dropna(inplace=True)
    
    return features_df


# Test feature creation
test_lags = [1, 3, 7]
test_windows = [7, 14]
test_features = create_time_series_features(df, test_lags, test_windows)

print(f"Original shape: {df.shape}")
print(f"Features shape: {test_features.shape}")
print(f"\nFeature columns:")
print(test_features.columns.tolist())
print(f"\nFirst 3 rows:")
print(test_features.head(3))

## 3️⃣ Train/Validation/Test Split

In [ ]:
def time_series_split(df, train_ratio=0.6, val_ratio=0.2):
    """
    Split time series into train/val/test sets.
    IMPORTANT: Must preserve temporal order!
    
    Arguments:
    df -- DataFrame with features
    train_ratio -- proportion for training
    val_ratio -- proportion for validation
    
    Returns:
    train_df, val_df, test_df
    """
    n = len(df)
    train_size = int(n * train_ratio)
    val_size = int(n * val_ratio)
    
    train_df = df.iloc[:train_size]
    val_df = df.iloc[train_size:train_size + val_size]
    test_df = df.iloc[train_size + val_size:]
    
    return train_df, val_df, test_df


# Create features and split
features_df = create_time_series_features(df, selected_lags=[1, 2, 3, 7, 14], 
                                          rolling_windows=[7, 14, 30])

train_df, val_df, test_df = time_series_split(features_df, train_ratio=0.6, val_ratio=0.2)

print("Dataset splits:")
print(f"Train: {len(train_df)} samples ({train_df.index[0]} to {train_df.index[-1]})")
print(f"Val:   {len(val_df)} samples ({val_df.index[0]} to {val_df.index[-1]})")
print(f"Test:  {len(test_df)} samples ({test_df.index[0]} to {test_df.index[-1]})")
print(f"Total: {len(train_df) + len(val_df) + len(test_df)} samples")

## 4️⃣ GA for Feature Selection and Hyperparameter Optimization

In [ ]:
def decode_timeseries_chromosome(chromosome, max_lag=14):
    """
    Decode chromosome for time series forecasting.
    
    Chromosome structure:
    [lag_selection (max_lag bits) | rolling_window_sizes (3 reals) | model_params (2 reals)]
    
    Arguments:
    chromosome -- encoded configuration
    max_lag -- maximum lag to consider
    
    Returns:
    selected_lags -- list of selected lag values
    rolling_windows -- list of rolling window sizes
    model_params -- dict of model hyperparameters
    """
    # Lag selection (first max_lag genes)
    lag_genes = chromosome[:max_lag]
    selected_lags = [i+1 for i, gene in enumerate(lag_genes) if gene > 0.5]
    
    # Ensure at least one lag
    if not selected_lags:
        selected_lags = [1]
    
    # Rolling window sizes (next 3 genes)
    window_genes = chromosome[max_lag:max_lag+3]
    # Scale to reasonable window sizes: 3 to 30 days
    rolling_windows = [int(3 + gene * 27) for gene in window_genes]
    rolling_windows = sorted(set(rolling_windows))  # Remove duplicates and sort
    
    # Model hyperparameters (next 2 genes for RandomForest)
    param_genes = chromosome[max_lag+3:max_lag+5]
    n_estimators = int(50 + param_genes[0] * 150)  # 50 to 200
    max_depth = int(3 + param_genes[1] * 17)  # 3 to 20
    
    model_params = {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'random_state': 42
    }
    
    return selected_lags, rolling_windows, model_params


# Test decoding
test_chromosome = np.array([0.8, 0.2, 0.9, 0.3, 0.1, 0.7, 0.9, 0.1, 0.2, 0.3, 0.4, 0.6, 0.7, 0.8,  # lags
                           0.3, 0.6, 0.9,  # windows
                           0.5, 0.7])  # model params

lags, windows, params = decode_timeseries_chromosome(test_chromosome, max_lag=14)
print("Test decoding:")
print(f"Selected lags: {lags}")
print(f"Rolling windows: {windows}")
print(f"Model params: {params}")

In [ ]:
def evaluate_timeseries_config(chromosome, df_original, max_lag=14, 
                               train_ratio=0.6, val_ratio=0.2, complexity_penalty=0.001):
    """
    Evaluate time series forecasting configuration.
    
    Arguments:
    chromosome -- encoded configuration
    df_original -- original time series DataFrame
    max_lag -- maximum lag value
    train_ratio, val_ratio -- split ratios
    complexity_penalty -- penalty for number of features
    
    Returns:
    fitness -- negative validation RMSE (higher is better)
    """
    try:
        # Decode chromosome
        selected_lags, rolling_windows, model_params = decode_timeseries_chromosome(
            chromosome, max_lag
        )
        
        # Create features
        features_df = create_time_series_features(
            df_original, selected_lags, rolling_windows, add_datetime_features=True
        )
        
        # Split data
        train_df, val_df, _ = time_series_split(features_df, train_ratio, val_ratio)
        
        if len(train_df) < 10 or len(val_df) < 5:
            return -1000.0  # Not enough data
        
        # Prepare X, y
        feature_cols = [col for col in features_df.columns if col != 'value']
        X_train = train_df[feature_cols]
        y_train = train_df['value']
        X_val = val_df[feature_cols]
        y_val = val_df['value']
        
        # Train model
        model = RandomForestRegressor(**model_params)
        model.fit(X_train, y_train)
        
        # Predict
        y_pred = model.predict(X_val)
        
        # Evaluate
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        
        # Complexity penalty
        n_features = len(feature_cols)
        complexity = complexity_penalty * n_features
        
        # Fitness (minimize RMSE, so negate it)
        fitness = -rmse - complexity
        
        return fitness
        
    except Exception as e:
        return -1000.0


# Test evaluation
test_fitness = evaluate_timeseries_config(test_chromosome, df, max_lag=14)
print(f"\nTest fitness: {test_fitness:.4f}")

### Run Genetic Algorithm

In [ ]:
def timeseries_genetic_algorithm(df_original, pop_size=40, max_generations=30,
                                 max_lag=14, mutation_rate=0.1, crossover_rate=0.8,
                                 elite_size=2):
    """
    Optimize time series forecasting with GA.
    
    Returns:
    best_chromosome, best_fitness, history
    """
    chromosome_length = max_lag + 3 + 2  # lags + windows + model_params
    
    # Initialize population
    population = np.random.rand(pop_size, chromosome_length)
    
    history = {
        'best_fitness': [],
        'mean_fitness': [],
        'best_rmse': [],
        'best_n_features': []
    }
    
    best_overall_fitness = -np.inf
    best_overall_chromosome = None
    
    print("Starting Time Series GA Optimization...\n")
    print(f"{'Gen':<6} {'Best Fit':<12} {'Best RMSE':<12} {'Mean Fit':<12}")
    print("="*60)
    
    for generation in range(max_generations):
        # Evaluate
        fitness = np.array([
            evaluate_timeseries_config(ind, df_original, max_lag)
            for ind in population
        ])
        
        # Track best
        best_idx = np.argmax(fitness)
        best_fitness = fitness[best_idx]
        best_chromosome = population[best_idx].copy()
        
        if best_fitness > best_overall_fitness:
            best_overall_fitness = best_fitness
            best_overall_chromosome = best_chromosome.copy()
        
        # Decode for logging
        lags, windows, _ = decode_timeseries_chromosome(best_chromosome, max_lag)
        n_features = len(lags) + 2*len(windows) + 4  # lags + rolling (mean+std) + datetime
        best_rmse = -best_fitness  # Approximate (includes penalty)
        
        history['best_fitness'].append(best_fitness)
        history['mean_fitness'].append(fitness.mean())
        history['best_rmse'].append(best_rmse)
        history['best_n_features'].append(n_features)
        
        if generation % 5 == 0 or generation == max_generations - 1:
            print(f"{generation:<6} {best_fitness:<12.4f} {best_rmse:<12.4f} {fitness.mean():<12.4f}")
        
        # Selection (Tournament)
        selected = []
        for _ in range(pop_size - elite_size):
            tournament_indices = np.random.choice(pop_size, 3, replace=False)
            winner_idx = tournament_indices[np.argmax(fitness[tournament_indices])]
            selected.append(population[winner_idx].copy())
        
        # Elitism
        elite_indices = np.argsort(fitness)[-elite_size:]
        elite = [population[i].copy() for i in elite_indices]
        
        # Crossover
        offspring = []
        for i in range(0, len(selected) - 1, 2):
            if np.random.rand() < crossover_rate:
                point = np.random.randint(1, chromosome_length)
                child1 = np.concatenate([selected[i][:point], selected[i+1][point:]])
                child2 = np.concatenate([selected[i+1][:point], selected[i][point:]])
                offspring.extend([child1, child2])
            else:
                offspring.extend([selected[i].copy(), selected[i+1].copy()])
        
        # Mutation
        for individual in offspring:
            for i in range(chromosome_length):
                if np.random.rand() < mutation_rate:
                    individual[i] += np.random.normal(0, 0.1)
                    individual[i] = np.clip(individual[i], 0, 1)
        
        population = np.array(elite + offspring[:pop_size - elite_size])
    
    print("="*60)
    print(f"\nOptimization complete! Best fitness: {best_overall_fitness:.4f}")
    
    return best_overall_chromosome, best_overall_fitness, history


# Run GA
best_chromosome, best_fitness, history = timeseries_genetic_algorithm(
    df, pop_size=30, max_generations=25, max_lag=14
)

## 5️⃣ Results and Evaluation

In [ ]:
# Decode best solution
best_lags, best_windows, best_model_params = decode_timeseries_chromosome(best_chromosome, max_lag=14)

print("\n" + "="*60)
print("BEST TIME SERIES CONFIGURATION")
print("="*60)
print(f"\nLag features: {best_lags}")
print(f"Rolling windows: {best_windows}")
print(f"Model parameters: {best_model_params}")

# Create features with best configuration
best_features_df = create_time_series_features(df, best_lags, best_windows, add_datetime_features=True)
train_df, val_df, test_df = time_series_split(best_features_df, train_ratio=0.6, val_ratio=0.2)

feature_cols = [col for col in best_features_df.columns if col != 'value']
X_train = train_df[feature_cols]
y_train = train_df['value']
X_val = val_df[feature_cols]
y_val = val_df['value']
X_test = test_df[feature_cols]
y_test = test_df['value']

# Train final model
final_model = RandomForestRegressor(**best_model_params)
final_model.fit(X_train, y_train)

# Predictions
y_pred_train = final_model.predict(X_train)
y_pred_val = final_model.predict(X_val)
y_pred_test = final_model.predict(X_test)

# Metrics
def calculate_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return rmse, mae, r2, mape

train_rmse, train_mae, train_r2, train_mape = calculate_metrics(y_train, y_pred_train)
val_rmse, val_mae, val_r2, val_mape = calculate_metrics(y_val, y_pred_val)
test_rmse, test_mae, test_r2, test_mape = calculate_metrics(y_test, y_pred_test)

print(f"\n{'Set':<10} {'RMSE':<10} {'MAE':<10} {'R²':<10} {'MAPE (%)':<10}")
print("-"*60)
print(f"{'Train':<10} {train_rmse:<10.4f} {train_mae:<10.4f} {train_r2:<10.4f} {train_mape:<10.2f}")
print(f"{'Val':<10} {val_rmse:<10.4f} {val_mae:<10.4f} {val_r2:<10.4f} {val_mape:<10.2f}")
print(f"{'Test':<10} {test_rmse:<10.4f} {test_mae:<10.4f} {test_r2:<10.4f} {test_mape:<10.2f}")

### Baseline Comparison

In [ ]:
# Baseline 1: Simple lag-1 model
simple_features_df = create_time_series_features(df, selected_lags=[1], 
                                                  rolling_windows=[], 
                                                  add_datetime_features=False)
train_simple, val_simple, test_simple = time_series_split(simple_features_df, 0.6, 0.2)

baseline_model = RandomForestRegressor(n_estimators=100, random_state=42)
baseline_model.fit(train_simple[['lag_1']], train_simple['value'])
baseline_pred = baseline_model.predict(test_simple[['lag_1']])
baseline_rmse, _, baseline_r2, _ = calculate_metrics(test_simple['value'], baseline_pred)

# Baseline 2: Persistence (naive forecast)
persistence_pred = test_df['value'].shift(1).dropna()
persistence_true = test_df['value'].iloc[1:]
persistence_rmse = np.sqrt(mean_squared_error(persistence_true, persistence_pred))

print("\n" + "="*60)
print("COMPARISON WITH BASELINES")
print("="*60)
print(f"\n{'Method':<30} {'Test RMSE':<15} {'Test R²':<15}")
print("-"*60)
print(f"{'Persistence (Naive)':<30} {persistence_rmse:<15.4f} {'N/A':<15}")
print(f"{'Simple Lag-1 RF':<30} {baseline_rmse:<15.4f} {baseline_r2:<15.4f}")
print(f"{'GA-Optimized':<30} {test_rmse:<15.4f} {test_r2:<15.4f}")
print("-"*60)

improvement = (baseline_rmse - test_rmse) / baseline_rmse * 100
print(f"\nGA improvement over Simple Lag-1: {improvement:.2f}%")

## 6️⃣ Visualization

In [ ]:
# Plot evolution history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fitness evolution
ax = axes[0]
ax.plot(history['best_fitness'], 'b-', linewidth=2, label='Best Fitness')
ax.plot(history['mean_fitness'], 'g--', linewidth=1.5, alpha=0.7, label='Mean Fitness')
ax.set_xlabel('Generation', fontsize=11)
ax.set_ylabel('Fitness', fontsize=11)
ax.set_title('Fitness Evolution', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# RMSE evolution
ax = axes[1]
ax.plot(history['best_rmse'], 'r-', linewidth=2)
ax.set_xlabel('Generation', fontsize=11)
ax.set_ylabel('Best RMSE (approx)', fontsize=11)
ax.set_title('RMSE Evolution', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot predictions vs actual
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Train set
ax = axes[0]
ax.plot(train_df.index, y_train, label='Actual', linewidth=1.5, alpha=0.7)
ax.plot(train_df.index, y_pred_train, label='Predicted', linewidth=1.5, alpha=0.7)
ax.set_ylabel('Value', fontsize=11)
ax.set_title(f'Training Set (RMSE={train_rmse:.2f}, R²={train_r2:.4f})', 
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Validation set
ax = axes[1]
ax.plot(val_df.index, y_val, label='Actual', linewidth=1.5, alpha=0.7)
ax.plot(val_df.index, y_pred_val, label='Predicted', linewidth=1.5, alpha=0.7)
ax.set_ylabel('Value', fontsize=11)
ax.set_title(f'Validation Set (RMSE={val_rmse:.2f}, R²={val_r2:.4f})', 
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Test set
ax = axes[2]
ax.plot(test_df.index, y_test, label='Actual', linewidth=1.5, alpha=0.7, color='blue')
ax.plot(test_df.index, y_pred_test, label='Predicted', linewidth=1.5, alpha=0.7, color='red')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Value', fontsize=11)
ax.set_title(f'Test Set (RMSE={test_rmse:.2f}, R²={test_r2:.4f})', 
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(range(len(feature_importance)), feature_importance['importance'], color='steelblue')
plt.yticks(range(len(feature_importance)), feature_importance['feature'])
plt.xlabel('Importance', fontsize=11)
plt.title('Feature Importance (GA-Optimized Model)', fontsize=12, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 5 most important features:")
print(feature_importance.head())

## 📊 Summary and Key Takeaways

### What We Learned

1. **Time Series Feature Engineering**:
   - Lag features capture temporal dependencies
   - Rolling statistics smooth noise and capture trends
   - Datetime features capture seasonality
   - Feature selection significantly impacts performance

2. **GA for Time Series**:
   - Simultaneously optimizes lag selection, window sizes, and model hyperparameters
   - Automates feature engineering process
   - Finds non-obvious feature combinations
   - Balances model complexity and accuracy

3. **Best Practices**:
   - **Temporal split**: Never shuffle time series data!
   - **Walk-forward validation**: Use proper time-based CV
   - **Baseline comparison**: Always compare against naive forecasts
   - **Interpretability**: Check feature importance to understand model

### Performance Insights

Typical GA improvements:
- **vs Persistence**: 30-60% RMSE reduction
- **vs Simple Lag-1**: 10-25% RMSE reduction
- **Feature reduction**: 30-50% fewer features with similar/better accuracy

### When to Use GA for Time Series

**Good fit**:
- Many potential lag features
- Unclear which lags are important
- Need automated feature engineering
- Have computational resources for GA search

**Challenges**:
- Computationally expensive
- Requires careful validation to avoid overfitting
- May not beat domain expertise on simple problems

### Extensions to Explore

1. **Multi-step forecasting**: Predict multiple time steps ahead
2. **Exogenous variables**: Include external features (weather, holidays, etc.)
3. **Ensemble forecasting**: Combine multiple GA-optimized models
4. **ARIMA/SARIMA optimization**: Use GA to find optimal (p,d,q) parameters
5. **Neural architecture search**: Optimize LSTM/GRU architectures with GA
6. **Online learning**: Adapt model as new data arrives

---

## 🎓 Exercises

1. **Multi-step ahead**: Modify to predict 7 days ahead instead of 1
2. **Real dataset**: Apply to stock prices, weather, or energy consumption
3. **Different models**: Try GradientBoosting, XGBoost, or LSTM
4. **Walk-forward CV**: Implement proper time series cross-validation
5. **Multi-objective**: Optimize accuracy AND forecast confidence intervals
6. **Ensemble**: Combine multiple GA-optimized configurations

---

**You've mastered GA-based time series optimization! This powerful technique automates feature engineering and hyperparameter tuning for temporal data.** 📈
